# Single-/two-exon model burden with and without ULC/Kinnex models (HPC)

Goal: quantify whether CAT's additional model burden is enriched for one- or two-exon transcript models, and repeat after excluding `unknown_likely_coding` CAT models.

This notebook expects per-assembly `gff_compare/tx_metrics.tsv` outputs from `compare_gff_features.py`.

Expected copy-back folder after completion: `results/intermediate_spreadsheets/exon_count_ulc_sensitivity/`.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))
QC_DIR = OUTPUT_DIR / 'qc_metrics'
WORK_DIR = OUTPUT_DIR / 'intermediate_spreadsheets' / 'exon_count_ulc_sensitivity'
FIG_DIR = WORK_DIR / 'figures'
WORK_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('OUTPUT_DIR:', OUTPUT_DIR)
print('QC_DIR:', QC_DIR, QC_DIR.exists())
print('WORK_DIR:', WORK_DIR)


In [ ]:
# Locate tx_metrics files.
tx_files = sorted(QC_DIR.rglob('gff_compare/tx_metrics.tsv'))
if not tx_files:
    tx_files = sorted(QC_DIR.rglob('tx_metrics.tsv'))
print('tx_metrics files:', len(tx_files))
print('\n'.join(map(str, tx_files[:5])))
if not tx_files:
    raise FileNotFoundError('No tx_metrics.tsv files found. Need GFF_COMPARE outputs from the pipeline.')


In [ ]:
# Aggregate in a memory-light per-file loop.
USECOLS = ['assembly_accession','sample_name','source','gene_id','transcript_id','biotype','biotype_group','n_exons','tx_len_bp','cds_len_bp']
ULC_VALUE = 'unknown_likely_coding'

def exon_class(n):
    try:
        n = int(n)
    except Exception:
        return 'unknown'
    if n == 1:
        return '1 exon'
    if n == 2:
        return '2 exons'
    if n >= 3:
        return '>=3 exons'
    return 'unknown'

rows = []
for i, fp in enumerate(tx_files, 1):
    if i == 1 or i % 50 == 0:
        print(f'{i}/{len(tx_files)} {fp}', flush=True)
    df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in USECOLS)
    df['n_exons'] = pd.to_numeric(df['n_exons'], errors='coerce')
    df['exon_class'] = df['n_exons'].map(exon_class)
    df['biotype_group'] = df.get('biotype_group', df['biotype']).fillna('unknown')
    df['biotype'] = df['biotype'].fillna('unknown')

    for analysis_set, sub in [
        ('all_models', df),
        ('cat_excluding_unknown_likely_coding', df[~((df['source'] == 'CAT') & (df['biotype'] == ULC_VALUE))].copy()),
    ]:
        g = (sub.groupby(['assembly_accession','sample_name','source','biotype_group','exon_class'])
             .size().reset_index(name='n_transcripts'))
        g['analysis_set'] = analysis_set
        rows.append(g)

counts = pd.concat(rows, ignore_index=True)
counts.to_csv(WORK_DIR / 'exon_class_counts_per_assembly.tsv', sep='\t', index=False)
print(counts.head().to_string(index=False))


In [ ]:
# Median counts and percentage composition by assembly.
counts = pd.read_csv(WORK_DIR / 'exon_class_counts_per_assembly.tsv', sep='\t')
EXON_ORDER = ['1 exon','2 exons','>=3 exons','unknown']
SOURCE_ORDER = ['Ensembl','CAT']

# Percentages within assembly/source/biotype_group/analysis_set.
totals = counts.groupby(['analysis_set','assembly_accession','source','biotype_group'])['n_transcripts'].transform('sum')
counts['pct_within_source_biotype_assembly'] = np.where(totals > 0, 100 * counts['n_transcripts'] / totals, np.nan)
counts.to_csv(WORK_DIR / 'exon_class_counts_and_percentages_per_assembly.tsv', sep='\t', index=False)

med_counts = (counts.groupby(['analysis_set','source','biotype_group','exon_class'])['n_transcripts']
              .median().reset_index(name='median_n_transcripts_per_assembly'))
med_pct = (counts.groupby(['analysis_set','source','biotype_group','exon_class'])['pct_within_source_biotype_assembly']
           .median().reset_index(name='median_pct_within_source_biotype'))
summary = med_counts.merge(med_pct, on=['analysis_set','source','biotype_group','exon_class'], how='outer')
summary.to_csv(WORK_DIR / 'exon_class_median_summary.tsv', sep='\t', index=False)
display(summary.head())


In [ ]:
# Render compact plots for protein_coding, lncRNA, pseudogene, and other_ncRNA.
summary = pd.read_csv(WORK_DIR / 'exon_class_median_summary.tsv', sep='\t')
BIOTYPES = ['protein_coding','lncRNA','pseudogene','other_ncRNA','other']
ANALYSES = ['all_models','cat_excluding_unknown_likely_coding']
EXON_ORDER = ['1 exon','2 exons','>=3 exons']
COLORS = {'1 exon':'#e74c3c', '2 exons':'#f39c12', '>=3 exons':'#3498db'}

for metric, ylabel, fname in [
    ('median_n_transcripts_per_assembly', 'Median transcripts per assembly', 'exon_class_median_counts'),
    ('median_pct_within_source_biotype', 'Median % within source/biotype', 'exon_class_median_percentages'),
]:
    fig, axes = plt.subplots(len(ANALYSES), len(BIOTYPES), figsize=(16, 6), sharey='row')
    for r, analysis in enumerate(ANALYSES):
        for c, bio in enumerate(BIOTYPES):
            ax = axes[r, c]
            sub = summary[(summary['analysis_set'] == analysis) & (summary['biotype_group'] == bio)]
            x = np.arange(2)
            width = 0.22
            for j, ex in enumerate(EXON_ORDER):
                vals = []
                for source in ['Ensembl','CAT']:
                    v = sub[(sub['source'] == source) & (sub['exon_class'] == ex)][metric]
                    vals.append(float(v.iloc[0]) if len(v) else 0.0)
                ax.bar(x + (j-1)*width, vals, width=width, color=COLORS[ex], label=ex)
            ax.set_title(bio, fontsize=9)
            ax.set_xticks(x); ax.set_xticklabels(['Ensembl','CAT'], rotation=25, ha='right')
            if c == 0:
                ax.set_ylabel(f'{analysis}\n{ylabel}')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=COLORS[e], label=e) for e in EXON_ORDER]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False)
    plt.tight_layout(rect=[0,0.08,1,1])
    fig.savefig(FIG_DIR / f'{fname}.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{fname}.pdf', bbox_inches='tight')
    plt.show()

print('Copy back:', WORK_DIR)
